In [4]:
!pip install -U "transformers[serving]" accelerate bitsandbytes sentence-transformers chromadb pypdf python-docx openai

In [5]:
from huggingface_hub import login
login()

In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "speakleash/Bielik-1.5B-v3.0-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    use_fast=False,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float32,
    device_map="cpu",
    low_cpu_mem_usage=True,
)

print("Model załadowany na CPU!")
print("Tokenizer:", type(tokenizer).__name__)
print("Urządzenie:", model.device)

c:\Studia\Przetwarzanie tekstu\myenv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Mateusz\.cache\huggingface\hub\models--speakleash--Bielik-1.5B-v3.0-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Studia\Przetwarzanie tekstu\myenv\Lib\site-packages\huggingface_hub\file_download.py:149: 

Model załadowany na CPU!
Tokenizer: LlamaTokenizer
Urządzenie: cpu


In [78]:
import json
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer
INDEX_PATH = "regulamin.faiss"
METADATA_PATH = "fragmenty_regulaminu.json"

EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL,
    device="cpu",
)

index = faiss.read_index(INDEX_PATH)

with open(METADATA_PATH, "r", encoding="utf-8") as file:
    chunks = json.load(file)


def retrieve_context(question, top_k=4):
    # Zamiana pytania na wektor tym samym modelem
    query_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    query_embedding = np.asarray(
        query_embedding,
        dtype=np.float32,
    )

    # Wyszukanie najbardziej podobnych chunków
    scores, indices = index.search(query_embedding, top_k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        if idx < 0:
            continue

        chunk = chunks[int(idx)]

        results.append({
            "paragraf": chunk.get("paragraf"),
            "ustep": chunk.get("ustep"),
            "text": chunk["text"],
            "score": float(score),
        })

    # Budowa kontekstu do promptu
    context_parts = []

    for result in results:
        location = f"§ {result['paragraf']}"

        location += f", ust. {result['ustep']}"

        context_parts.append(
            f"Źródło:[{location}] \n {result['text']}"
        )

    context = "\n\n".join(context_parts)

    return context, results

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1686.42it/s]


In [82]:
def generate_answer(question, context=""):
    messages = [
        {
            "role": "system",
            "content": (
                """Jesteś asystentem odpowiadającym po polsku na pytania dotyczące regulaminu studiów.
                    Zasady odpowiedzi:
                    1. Odpowiadaj wyłącznie na podstawie informacji zawartych w dostarczonym kontekście regulaminu.
                    2. Każdą odpowiedź merytoryczną poprzyj wskazaniem źródła: numeru paragrafu (§) oraz ustępu, z którego pochodzi informacja.
                    3. Zachowuj dokładną numerację paragrafów i ustępów podaną w kontekście. Nie wymyślaj ani nie uzupełniaj numerów na podstawie własnej wiedzy.
                    4. Jeżeli odpowiedź wynika z kilku fragmentów, wskaż wszystkie odpowiednie paragrafy i ustępy.
                    5. Jeżeli kontekst nie zawiera wystarczających informacji, aby odpowiedzieć na pytanie, poinformuj: „Brak wystarczających informacji w dostarczonym kontekście regulaminu”.
                    6. Jeżeli fragment nie zawiera numeru paragrafu lub ustępu, nie zgaduj go. Wyraźnie zaznacz, że nie można ustalić pełnego źródła na podstawie dostarczonych danych.
                    7. Odpowiadaj konkretnie i zwięźle. Oddziel odpowiedź od wskazania źródła."""
                ),
        },
        {
            "role": "user",
            "content": (
                f"Kontekst regulaminu:\n{context}\n\n"
                f"Pytanie:\n{question}"
            ),
        },
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = outputs[
        0, inputs["input_ids"].shape[-1]:
    ]

    return tokenizer.decode(
        new_tokens,
        skip_special_tokens=True,
    ).strip()

In [83]:
question = "Jakie wymogi muszą zostać spełnione aby student mógł podejść do egzaminu dyplomowego?"
context, results = retrieve_context(
        question,
        top_k=5,
    )

print(generate_answer(question, context))

Aby student mógł podejść do egzaminu dyplomowego, muszą zostać spełnione następujące wymogi:

1. **Uzyskanie wymaganej programem studiów liczby punktów ECTS** (§ 30, ust. 10).
2. **Uzyskanie pozytywnej oceny ze wszystkich zajęć w ramach kształcenia przewidzianych programem studiów** (§ 30, ust. 10).
3. **Spełnienie innych wymagań programowych, w tym zaliczenie praktyk zawodowych** (§ 30, ust. 10).
4. **Uzyskanie pozytywnej oceny promotora i recenzenta z pracy dyplomowej** (§ 30, ust. 9).
5. **Pozytywna weryfikacja pracy dyplomowej pod kątem braku naruszeń praw autorskich i praw pokrewnych** (§ 30, ust. 9).

Dodatkowo, student musi mieć zaliczone zajęcia dydaktyczne wchodzące w skład danego przedmiotu, warunkujące przystąpienie do egzaminu (§ 18, ust. 7).
